# Batch Full-Text Extraction Evaluation

Evaluate full-text extraction using GROBID-parsed PDFs across all Fuster validation samples.

**Approach:**
- Parse PDFs with GROBID to extract all sections
- Build full-text prompts using all sections
- Run GPT-5-mini extraction with low reasoning
- Compare against manually annotated ground truth
- Use `DatasetFeaturesNormalized` for ground truth validation

In [14]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# Project modules
from llm_metadata.fulltext_pipeline import (
    FulltextPipelineConfig,
    SectionSelectionConfig,
    GPTClassifyConfig,
    FulltextInputRecord,
    FulltextOutputRecord,
    ParsedDocumentRecord,
    PromptRecord,
    # Combined flow
    fulltext_extraction_pipeline,
    # Staged flows (separate parallelization)
    staged_fulltext_pipeline,
    grobid_parsing_flow,
    prompt_building_flow,
    gpt_classification_flow,
    # Single PDF
    process_single_pdf,
    # Utilities
    load_input_manifest,
    save_output_manifest,
    build_fulltext_prompt,
    collect_relevant_sections,
)
from llm_metadata.pdf_parsing import process_pdf, ParsedDocument, Section
from llm_metadata.gpt_classify import classify_abstract
from llm_metadata.chunking import count_tokens
from llm_metadata.section_normalize import classify_section
from llm_metadata.schemas.chunk_metadata import SectionType
from llm_metadata.schemas.fuster_features import (
    DatasetFeatures,
    DatasetFeaturesNormalized,
)
from llm_metadata.schemas.evaluation import (
    evaluate_indexed,
    EvaluationConfig,
    FuzzyMatchConfig,
    micro_average,
    macro_f1,
)

# Paths
PDF_DIR = Path("data/pdfs/fuster")
MANIFEST_PATH = PDF_DIR / "manifest.csv"
GROUND_TRUTH_PATH = Path("data/dataset_092624.xlsx")
OUTPUT_DIR = Path("artifacts/fulltext_results")

print(f"PDF directory: {PDF_DIR.absolute()}")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")
print(f"Ground truth exists: {GROUND_TRUTH_PATH.exists()}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PDF directory: c:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster
Manifest exists: True
Ground truth exists: True


## Step 1: Load Manifest and Ground Truth

Map dataset DOIs to article DOIs and load ground truth annotations.

In [15]:
# Load manifest with DOI mapping
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest: {len(manifest_df)} total rows")

# Filter to downloaded PDFs only
downloaded_df = manifest_df[manifest_df['status'] == 'downloaded'].copy()
print(f"Downloaded PDFs: {len(downloaded_df)}")

# Create DOI mapping (dataset_doi -> article_doi)
doi_mapping = dict(zip(downloaded_df['dataset_doi'], downloaded_df['article_doi']))
print(f"DOI mappings: {len(doi_mapping)}")

# Display sample
downloaded_df[['article_doi', 'dataset_doi', 'downloaded_pdf_path']].head(5)

Manifest: 75 total rows
Downloaded PDFs: 70
DOI mappings: 70


,article_doi,dataset_doi,downloaded_pdf_path
1,10.1093/jhered/esx103,10.5061/dryad.121sk,fuster\10.1093_jhered_esx103.pdf
2,10.1371/journal.pone.0128238,10.5061/dryad.1771t,fuster\10.1371_journal.pone.0128238.pdf
3,10.1371/journal.pone.0073695,10.5061/dryad.1cc4v,fuster\10.1371_journal.pone.0073695.pdf
4,10.1002/ece3.4685,10.5061/dryad.24q6q70,fuster\10.1002_ece3.4685.pdf
5,10.1639/0007-2745-119.1.008,10.5061/dryad.24rj8,fuster\10.1639_0007-2745-119.1.008.pdf


In [16]:
# Load ground truth
gt_df = pd.read_excel(GROUND_TRUTH_PATH)
print(f"Ground truth: {len(gt_df)} total rows")

# Extract dataset DOI from URL column
gt_df['dataset_doi'] = gt_df['url'].str.replace('https://doi.org/', '', regex=False)

# Filter to DOIs that have downloaded PDFs
gt_matched = gt_df[gt_df['dataset_doi'].isin(doi_mapping.keys())].copy()
print(f"Ground truth with matched PDFs: {len(gt_matched)}")

# Add article DOI mapping
gt_matched['article_doi'] = gt_matched['dataset_doi'].map(doi_mapping)

gt_matched[['dataset_doi', 'article_doi', 'data_type', 'species']].head(5)

Ground truth: 418 total rows
Ground truth with matched PDFs: 70


,dataset_doi,article_doi,data_type,species
2,10.5061/dryad.121sk,10.1093/jhered/esx103,EBV genetic analysis,Glyptemys insculpta
4,10.5061/dryad.1771t,10.1371/journal.pone.0128238,density,"raccoons, striped skunks"
6,10.5061/dryad.1cc4v,10.1371/journal.pone.0073695,presence only,Rangifer tarandus caribou
7,10.5061/dryad.24q6q70,10.1002/ece3.4685,other,Rangifer tarandus caribou
8,10.5061/dryad.24rj8,10.1639/0007-2745-119.1.008,genetic analysis,Aspicilia bicensis


In [17]:
# Validate ground truth with DatasetFeaturesNormalized
RELEVANT_COLS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset'
]

ground_truth_by_dataset = {}
validation_errors = []

for _, row in gt_matched.iterrows():
    dataset_doi = row['dataset_doi']
    try:
        # Extract relevant columns and convert to dict
        row_dict = {col: row[col] for col in RELEVANT_COLS if col in row.index}
        
        # Validate using DatasetFeaturesNormalized
        validated = DatasetFeaturesNormalized.model_validate(row_dict)
        ground_truth_by_dataset[dataset_doi] = validated
    except Exception as e:
        validation_errors.append({'doi': dataset_doi, 'error': str(e)})

print(f"Successfully validated: {len(ground_truth_by_dataset)} records")
print(f"Validation errors: {len(validation_errors)} records")

if validation_errors:
    print("\nFirst 3 errors:")
    for err in validation_errors[:3]:
        print(f"  {err['doi']}: {err['error'][:100]}...")

Successfully validated: 70 records
Validation errors: 0 records


## Step 2: Configure Pipeline

Set up full-text extraction with all sections, gpt-5-mini, and low reasoning.

In [18]:
# Pipeline configuration
config = FulltextPipelineConfig(
    section_config=SectionSelectionConfig(
        include_all=True,  # Use ALL sections from PDFs
        include_abstract=True,
    ),
    gpt_config=GPTClassifyConfig(
        model="gpt-5-mini",
        reasoning={"effort": "low"},
        max_output_tokens=4096,
    ),
    grobid_url="http://localhost:8070",
    pdf_dir=PDF_DIR,
    text_format=DatasetFeatures,
)

print("Pipeline Configuration:")
print(f"  Model: {config.gpt_config.model}")
print(f"  Reasoning: {config.gpt_config.reasoning}")
print(f"  Section selection: include_all={config.section_config.include_all}")
print(f"  GROBID URL: {config.grobid_url}")

Pipeline Configuration:
  Model: gpt-5-mini
  Reasoning: {'effort': 'low'}
  Section selection: include_all=True
  GROBID URL: http://localhost:8070


## Step 3: Run Full-Text Extraction

Process all PDFs through GROBID parsing and GPT classification.

In [19]:
# Build input records for PDFs with ground truth
input_records = []
for _, row in gt_matched.iterrows():
    dataset_doi = row['dataset_doi']
    article_doi = row['article_doi']
    
    # Find PDF path from manifest
    manifest_row = downloaded_df[downloaded_df['dataset_doi'] == dataset_doi]
    if manifest_row.empty:
        continue
    
    pdf_path = PDF_DIR / manifest_row.iloc[0]['downloaded_pdf_path']
    if not pdf_path.exists():
        # Try alternate path construction
        pdf_path = PDF_DIR.parent / manifest_row.iloc[0]['downloaded_pdf_path']
    
    if pdf_path.exists():
        input_records.append(FulltextInputRecord(
            article_doi=article_doi,
            dataset_doi=dataset_doi,
            pdf_path=str(pdf_path),
        ))

print(f"Input records to process: {len(input_records)}")

Input records to process: 70


In [20]:
# Process PDFs using STAGED PIPELINE with separate parallelization
# Stage 1: GROBID parsing (max_workers=10) - GROBID handles concurrency well
# Stage 2: Prompt building (max_workers=5) - CPU-bound
# Stage 3: GPT classification (max_workers=5) - API rate limits

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_manifest_path = OUTPUT_DIR / f"fulltext_results_{timestamp}.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running STAGED pipeline on {len(input_records)} PDFs...")
print(f"Using {config.gpt_config.model} with {config.gpt_config.reasoning}")
print()

# Run the staged Prefect flow with separate parallelization per stage
results = staged_fulltext_pipeline(
    input_records=input_records,
    config=config,
    output_manifest=output_manifest_path,
    grobid_workers=10,  # Higher parallelization for GROBID
    gpt_workers=5,       # Controlled parallelization for GPT API
)

# Summary
success_count = sum(1 for r in results if r.status == "success")
error_count = sum(1 for r in results if r.status == "error")
skipped_count = sum(1 for r in results if r.status == "skipped")
print(f"\nProcessing complete: {success_count} success, {error_count} errors, {skipped_count} skipped")
print(f"Results saved to: {output_manifest_path}")

Running STAGED pipeline on 70 PDFs...
Using gpt-5-mini with {'effort': 'low'}



14:30:25.840 | INFO    | Flow run 'tuscan-hare' - Beginning flow run 'tuscan-hare' for flow 'staged-fulltext-pipeline'

Starting staged pipeline for 70 PDFs
  Stage 1: GROBID parsing (max_workers=10)
  Stage 2: Prompt building
  Stage 3: GPT classification (max_workers=5)



14:30:25.971 | INFO    | Flow run 'masked-gazelle' - Beginning subflow run 'masked-gazelle' for flow 'grobid-parsing-flow'

Parsing 70 PDFs with GROBID (max_workers=10)...


...

14:30:35.376 | INFO    | Task run 'classify_prompt_record-629' - Finished in state Completed()

GPT classification complete: 45 success, 0 errors, 25 skipped
Total cost: $0.3838
Saved output manifest to artifacts\fulltext_results\fulltext_results_20260111_143025.csv (70 records)


14:30:35.526 | INFO    | Flow run 'prudent-pudu' - Finished in state Completed()

14:30:35.571 | INFO    | Flow run 'tuscan-hare' - Finished in state Completed()


Processing complete: 45 success, 0 errors, 25 skipped
Results saved to: artifacts\fulltext_results\fulltext_results_20260111_143025.csv


## Step 4: Prepare Extractions for Evaluation

Convert extraction results to Pydantic models for evaluation.

In [21]:
# Build predictions by dataset DOI
predictions_by_dataset = {}

for result in results:
    if result.status != "success" or not result.extraction:
        continue
    
    dataset_doi = result.dataset_doi
    if not dataset_doi:
        continue
    
    try:
        # Create DatasetFeatures from extraction dict
        pred = DatasetFeatures.model_validate(result.extraction)
        predictions_by_dataset[dataset_doi] = pred
    except Exception as e:
        print(f"Error validating prediction for {dataset_doi}: {e}")

print(f"Valid predictions: {len(predictions_by_dataset)}")

# Find common DOIs between predictions and ground truth
common_dois = set(predictions_by_dataset.keys()) & set(ground_truth_by_dataset.keys())
print(f"Common DOIs for evaluation: {len(common_dois)}")

Valid predictions: 45
Common DOIs for evaluation: 45


## Step 5: Evaluate Full-Text Extraction

Compare full-text extractions against ground truth using evaluation framework.

In [43]:
# Evaluation fields
EVAL_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f', 'species'
]

# Evaluation configuration with fuzzy matching for species
eval_config = EvaluationConfig(
    casefold_strings=True,
    treat_lists_as_sets=True,
    fuzzy_match_fields={
        'species': FuzzyMatchConfig(threshold=80),
    }
)

# Filter to common DOIs
true_by_id = {doi: ground_truth_by_dataset[doi] for doi in common_dois}
pred_by_id = {doi: predictions_by_dataset[doi] for doi in common_dois}

# Run evaluation
fulltext_report = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=EVAL_FIELDS,
    config=eval_config,
)

print("FULL-TEXT Extraction Metrics:")
print("=" * 70)
display(fulltext_report.metrics_to_pandas())

FULL-TEXT Extraction Metrics:


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,27,108,28,0,45,0.200000,0.490909,0.284211,0.600000,0.022222
1,geospatial_info_dataset,16,168,8,0,45,0.086957,0.666667,0.153846,0.355556,0.000000
2,spatial_range_km2,19,6,13,10,45,0.760000,0.593750,0.666667,0.644444,0.644444
6,species,27,211,43,0,45,0.113445,0.385714,0.175325,0.600000,0.066667
5,temp_range_f,27,7,10,8,45,0.794118,0.729730,0.760563,0.777778,0.777778
4,temp_range_i,29,6,8,8,45,0.828571,0.783784,0.805556,0.822222,0.822222
3,temporal_range,4,36,33,4,45,0.100000,0.108108,0.103896,0.177778,0.177778


In [35]:
# Aggregate metrics
ft_micro = micro_average(fulltext_report.field_metrics.values())
ft_macro = macro_f1(fulltext_report.field_metrics.values())

print("\nAggregate Metrics:")
print("=" * 50)
print(f"{'Metric':<25} {'Value':>15}")
print("-" * 50)
print(f"{'Micro Precision':<25} {ft_micro.precision or 0:>15.3f}")
print(f"{'Micro Recall':<25} {ft_micro.recall or 0:>15.3f}")
print(f"{'Micro F1':<25} {ft_micro.f1 or 0:>15.3f}")
print(f"{'Macro F1':<25} {ft_macro or 0:>15.3f}")
print(f"{'Records Evaluated':<25} {len(common_dois):>15}")
print("=" * 50)


Aggregate Metrics:
Metric                              Value
--------------------------------------------------
Micro Precision                     0.255
Micro Recall                        0.589
Micro F1                            0.356
Macro F1                            0.445
Records Evaluated                      45


## Step 6: Per-Field Analysis

In [36]:
# Per-field metrics for full-text extraction
field_data = []

for field in EVAL_FIELDS:
    fm = fulltext_report.field_metrics.get(field)
    if fm:
        field_data.append({
            'field': field,
            'precision': round(fm.precision or 0, 3),
            'recall': round(fm.recall or 0, 3),
            'f1': round(fm.f1 or 0, 3),
            'tp': fm.tp,
            'fp': fm.fp,
            'fn': fm.fn,
            'exact_match_rate': round(fm.exact_match_rate or 0, 3),
        })

field_df = pd.DataFrame(field_data)
print("Per-Field Metrics:")
display(field_df)

# Identify best and worst performing fields
if len(field_data) > 0:
    best = max(field_data, key=lambda x: x['f1'])
    worst = min(field_data, key=lambda x: x['f1'])
    print(f"\nBest performing field: {best['field']} (F1={best['f1']})")
    print(f"Worst performing field: {worst['field']} (F1={worst['f1']})")

Per-Field Metrics:


,field,precision,recall,f1,tp,fp,fn,exact_match_rate
0,data_type,0.200,0.491,0.284,27,108,28,0.022
1,geospatial_info_dataset,0.087,0.667,0.154,16,168,8,0.000
2,spatial_range_km2,0.760,0.594,0.667,19,6,13,0.644
3,temporal_range,0.100,0.108,0.104,4,36,33,0.178
4,temp_range_i,0.829,0.784,0.806,29,6,8,0.822
5,temp_range_f,0.794,0.730,0.761,27,7,10,0.778
6,species,0.226,0.714,0.344,50,171,20,0.267



Best performing field: temp_range_i (F1=0.806)
Worst performing field: temporal_range (F1=0.104)


## Step 7: Cost Analysis

In [37]:
# Cost analysis from results
successful_results = [r for r in results if r.status == "success"]

if successful_results:
    total_cost = sum(r.cost_usd or 0 for r in successful_results)
    total_input_tokens = sum(r.input_tokens or 0 for r in successful_results)
    total_output_tokens = sum(r.output_tokens or 0 for r in successful_results)
    total_fulltext_tokens = sum(r.fulltext_tokens or 0 for r in successful_results)
    total_abstract_tokens = sum(r.abstract_tokens or 0 for r in successful_results)
    
    print("\nCOST ANALYSIS")
    print("=" * 50)
    print(f"{'Metric':<30} {'Value':>18}")
    print("-" * 50)
    print(f"{'Total PDFs Processed':<30} {len(successful_results):>18}")
    print(f"{'Avg Abstract Tokens':<30} {total_abstract_tokens / len(successful_results):>18,.0f}")
    print(f"{'Avg Full-text Tokens':<30} {total_fulltext_tokens / len(successful_results):>18,.0f}")
    print(f"{'Token Increase (avg)':<30} {(total_fulltext_tokens - total_abstract_tokens) / len(successful_results):>18,.0f}")
    print("-" * 50)
    print(f"{'Total Input Tokens':<30} {total_input_tokens:>18,}")
    print(f"{'Total Output Tokens':<30} {total_output_tokens:>18,}")
    print(f"{'Total Cost (USD)':<30} ${total_cost:>17.4f}")
    print(f"{'Avg Cost per PDF (USD)':<30} ${total_cost / len(successful_results):>17.5f}")
    print("=" * 50)


COST ANALYSIS
Metric                                      Value
--------------------------------------------------
Total PDFs Processed                           45
Avg Abstract Tokens                           315
Avg Full-text Tokens                        7,778
Token Increase (avg)                        7,463
--------------------------------------------------
Total Input Tokens                        390,363
Total Output Tokens                        76,530
Total Cost (USD)               $           0.3838
Avg Cost per PDF (USD)         $          0.00853


## Step 8: Mismatch Exploration

Explore prediction errors and mismatches for each field.

In [38]:
# Build comprehensive mismatch dataframe for exploration
mismatch_records = []

for doi in common_dois:
    gt = ground_truth_by_dataset[doi]
    pred = predictions_by_dataset[doi]
    
    gt_dict = gt.model_dump()
    pred_dict = pred.model_dump()
    
    for field in EVAL_FIELDS:
        gt_val = gt_dict.get(field)
        pred_val = pred_dict.get(field)
        
        # Normalize for comparison
        gt_str = str(gt_val) if gt_val is not None else ""
        pred_str = str(pred_val) if pred_val is not None else ""
        
        # Check if mismatch
        is_match = gt_str.lower() == pred_str.lower() if gt_str and pred_str else gt_str == pred_str
        
        # Determine if values are present (positive) or absent (negative)
        gt_present = bool(gt_val is not None and gt_str.strip())
        pred_present = bool(pred_val is not None and pred_str.strip())
        
        # Calculate confusion matrix values
        tp = 1 if gt_present and pred_present and is_match else 0
        fp = 1 if pred_present and (not gt_present or not is_match) else 0
        fn = 1 if gt_present and not pred_present else 0
        tn = 1 if not gt_present and not pred_present else 0
        
        mismatch_records.append({
            'dataset_doi': doi,
            'field': field,
            'ground_truth': gt_val,
            'prediction': pred_val,
            'match': is_match,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'tn': tn,
        })

mismatches_df = pd.DataFrame(mismatch_records)
print(f"Total comparisons: {len(mismatches_df)}")
print(f"Mismatches: {len(mismatches_df[~mismatches_df['match']])}")
print(f"Matches: {len(mismatches_df[mismatches_df['match']])}")

# Filter to mismatches only
mismatches_only = mismatches_df[~mismatches_df['match']].copy()

# Summary by field
print("\nMISMATCHES BY FIELD:")
print("=" * 50)
mismatch_summary = mismatches_only.groupby('field').size().sort_values(ascending=False)
display(mismatch_summary)

# View mismatches for a specific field (change field name to explore)
EXPLORE_FIELD = 'species'  # Change this to explore different fields

field_mismatches = mismatches_only[mismatches_only['field'] == EXPLORE_FIELD]
print(f"\n{EXPLORE_FIELD.upper()} MISMATCHES ({len(field_mismatches)} records):")
print("-" * 70)
display(field_mismatches[['dataset_doi', 'ground_truth', 'prediction', 'tp', 'fp', 'fn', 'tn']])

Total comparisons: 315
Mismatches: 203
Matches: 112

MISMATCHES BY FIELD:


field
geospatial_info_dataset    45
data_type                  44
species                    43
temporal_range             37
spatial_range_km2          16
temp_range_f               10
temp_range_i                8
dtype: int64


SPECIES MISMATCHES (43 records):
----------------------------------------------------------------------


,dataset_doi,ground_truth,prediction,tp,fp,fn,tn
6,10.5061/dryad.n3c2f,[caribou],"[woodland caribou Rangifer tarandus caribou, m...",0,1,0,0
13,10.5061/dryad.2n5h6,[black-legged tick],"[black-legged tick, Ixodes scapularis Say 1821]",0,1,0,0
20,10.5061/dryad.r6s1c,"[Dendragapus canadensis, Poecile hudsonicus, A...","[spruce grouse (Dendragapus canadensis), borea...",0,1,0,0
27,10.5061/dryad.24q6q70,[Rangifer tarandus caribou],"[woodland caribou, Rangifer tarandus caribou]",0,1,0,0
34,10.5061/dryad.796t8,[40 species (mostly birds and few mammals)],[greater snow goose (Chen caerulescens atlanti...,0,1,0,0
41,10.5061/dryad.fp659276,[Procyon lotor],"[raccoon (Procyon lotor), striped skunk (Mephi...",0,1,0,0
55,10.5061/dryad.bf771,"[wolf, moose]","[grey wolf (Canis lupus), woodland caribou (Ra...",0,1,0,0
62,10.5061/dryad.rp77j,[Phragmites australis],"[Phragmites australis, common reed, haplotype ...",0,1,0,0
76,10.5061/dryad.t11f5,[10 fish species],"[rock bass, cutlip minnow, creek chub, yellow ...",0,1,0,0
83,10.5061/dryad.4k739,[Atlantic salmon],[Atlantic salmon (Salmo salar)],0,1,0,0


In [39]:
# Explore a single DOI in detail
# Change EXPLORE_DOI to investigate specific records
EXPLORE_DOI = list(common_dois)[0]  # First DOI by default

print(f"DETAILED COMPARISON FOR: {EXPLORE_DOI}")
print("=" * 80)

gt = ground_truth_by_dataset[EXPLORE_DOI]
pred = predictions_by_dataset[EXPLORE_DOI]

gt_dict = gt.model_dump()
pred_dict = pred.model_dump()

for field in EVAL_FIELDS:
    gt_val = gt_dict.get(field)
    pred_val = pred_dict.get(field)
    match_str = "MATCH" if str(gt_val).lower() == str(pred_val).lower() else "MISMATCH"
    
    print(f"\n{field}:")
    print(f"  Ground Truth: {gt_val}")
    print(f"  Prediction:   {pred_val}")
    print(f"  Status:       [{match_str}]")

DETAILED COMPARISON FOR: 10.5061/dryad.n3c2f

data_type:
  Ground Truth: ['other']
  Prediction:   ['time_series']
  Status:       [MISMATCH]

geospatial_info_dataset:
  Ground Truth: None
  Prediction:   ['sample', 'range', 'maps']
  Status:       [MISMATCH]

spatial_range_km2:
  Ground Truth: 31000.0
  Prediction:   31000.0
  Status:       [MATCH]

temporal_range:
  Ground Truth: 2004-2010
  Prediction:   2004-2010
  Status:       [MATCH]

temp_range_i:
  Ground Truth: 2004
  Prediction:   2004
  Status:       [MATCH]

temp_range_f:
  Ground Truth: 2010
  Prediction:   2010
  Status:       [MATCH]

species:
  Ground Truth: ['caribou']
  Prediction:   ['woodland caribou Rangifer tarandus caribou', 'moose Alces alces', 'gray wolf Canis lupus', 'black bear Ursus americanus']
  Status:       [MISMATCH]


In [40]:
# List all DOIs with their mismatch counts for easy exploration
doi_mismatch_counts = mismatches_only.groupby('dataset_doi').size().sort_values(ascending=False)

print("DOIs RANKED BY MISMATCH COUNT:")
print("=" * 60)
print("(Use these DOIs to set EXPLORE_DOI in the cell above)")
print()

for doi, count in doi_mismatch_counts.items():
    mismatched_fields = mismatches_only[mismatches_only['dataset_doi'] == doi]['field'].tolist()
    print(f"{doi}: {count} mismatches")
    print(f"  Fields: {', '.join(mismatched_fields)}")
    print()

DOIs RANKED BY MISMATCH COUNT:
(Use these DOIs to set EXPLORE_DOI in the cell above)

10.5061/dryad.36634: 7 mismatches
  Fields: data_type, geospatial_info_dataset, spatial_range_km2, temporal_range, temp_range_i, temp_range_f, species

10.5061/dryad.sf1k4m8: 7 mismatches
  Fields: data_type, geospatial_info_dataset, spatial_range_km2, temporal_range, temp_range_i, temp_range_f, species

10.5061/dryad.3nh72: 6 mismatches
  Fields: data_type, geospatial_info_dataset, spatial_range_km2, temporal_range, temp_range_i, species

10.5061/dryad.f7bq534: 6 mismatches
  Fields: data_type, geospatial_info_dataset, spatial_range_km2, temporal_range, temp_range_f, species

10.5061/dryad.b46k4: 6 mismatches
  Fields: data_type, geospatial_info_dataset, spatial_range_km2, temporal_range, temp_range_i, species

10.5061/dryad.fp659276: 6 mismatches
  Fields: data_type, geospatial_info_dataset, spatial_range_km2, temporal_range, temp_range_i, species

10.5061/dryad.n726pq6: 6 mismatches
  Fields: data_